<a href="https://colab.research.google.com/github/kmarkande/adaptive-memory-ai/blob/main/01_python_and_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
# ==========================================
# AMA DATABASE STARTUP
# ==========================================

import sqlite3
import os

# 🔴 1. Define database and backup locations
database_path = "/content/adaptive_memory.db"
backup_folder = "/content/drive/MyDrive/AdaptiveMemoryAI/backups"
backup_path = os.path.join(backup_folder, "adaptive_memory.db")


# 🔴 2. Define database validation function
def check_database(path):

    if not os.path.exists(path):
        return False

    try:
        connection = sqlite3.connect(
            f"file:{path}?mode=ro",
            uri=True
        )

        cursor = connection.cursor()

        cursor.execute("PRAGMA integrity_check")
        integrity = cursor.fetchone()[0]

        cursor.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        """)

        tables = {row[0] for row in cursor.fetchall()}

        connection.close()

        return (
            integrity == "ok"
            and {"entities", "episodes"}.issubset(tables)
        )

    except sqlite3.Error:
        return False


# 🔴 3. Validate active database or restore backup
if check_database(database_path):

    print("Active database is valid.")

elif check_database(backup_path):

    print("Active database missing or invalid.")
    print("Restoring from Google Drive...")

    temporary_restore = database_path + ".tmp"

    backup_connection = sqlite3.connect(
        f"file:{backup_path}?mode=ro",
        uri=True
    )

    restore_connection = sqlite3.connect(temporary_restore)

    backup_connection.backup(restore_connection)

    restore_connection.close()
    backup_connection.close()

    if check_database(temporary_restore):

        os.replace(temporary_restore, database_path)
        print("Database restored successfully!")

    else:

        os.remove(temporary_restore)
        raise RuntimeError("Restored database failed validation.")

else:

    raise RuntimeError(
        "No valid database or backup found. "
        "Stopping to prevent creation of an empty database."
    )


# 🔴 4. Connect only after validation or restoration
connection = sqlite3.connect(database_path)
cursor = connection.cursor()

print("AMA database connection established!")

Active database is valid.
AMA database connection established!
